# Exploring which reactions should be kept and removed in light of ubiquinol 10 pathway being used

In [4]:
import sys
from helpers import *
if ".." not in sys.path:
    sys.path.append("..")
    
import cobra
import pandas as pd
from cobra.io import read_sbml_model

model = read_sbml_model("../model/Rpom_05.xml")
model



Name,Rpom_05
Memory address,123e3fd90
Number of metabolites,1793
Number of reactions,1790
Number of genes,967
Number of groups,0
Objective expression,1.0*Rpom_hwa_biomass - 1.0*Rpom_hwa_biomass_reverse_5ec2f
Compartments,"c, p, e"


In [51]:
search_metabolites(["ubiquinol"])

id                                    | name                         | formula 
--------------------------------------+------------------------------+---------
CPD-9958[c]                           | ubiquinol-10                 | C59H92O4
UBIQUINOL-30[c]                       | ubiquinol-6                  | C39H60O4
CPD-9956[c]                           | ubiquinol-8                  | C49H76O4
OCTAPRENYL-METHYL-OH-METHOXY-BENZQ[c] | 3-demethylubiquinol-8        | C48H74O4
CPD-9873[c]                           | 3-demethylubiquinol-10       | C58H90O4
CPD-9861[c]                           | 3-demethylubiquinol-<i>7</i> | C43H66O4
CPD-9955[c]                           | ubiquinol-<i>7</i>           | C44H68O4


In [50]:
search_metabolites(["ubiquin"])

id                                    | name                         | formula 
--------------------------------------+------------------------------+---------
CPD-9958[c]                           | ubiquinol-10                 | C59H92O4
UBIQUINONE-10[c]                      | ubiquinone-10                | C59H90O4
UBIQUINOL-30[c]                       | ubiquinol-6                  | C39H60O4
UBIQUINONE-6[c]                       | ubiquinone-6                 | C39H58O4
CPD-9956[c]                           | ubiquinol-8                  | C49H76O4
OCTAPRENYL-METHYL-OH-METHOXY-BENZQ[c] | 3-demethylubiquinol-8        | C48H74O4
UBIQUINONE-8[c]                       | ubiquinone-8                 | C49H74O4
CPD-9873[c]                           | 3-demethylubiquinol-10       | C58H90O4
CPD-9861[c]                           | 3-demethylubiquinol-<i>7</i> | C43H66O4
CPD-9955[c]                           | ubiquinol-<i>7</i>           | C44H68O4


In [53]:
# begin by looking at non-ubiquinol 10 reactions
notubi10 = ["UBIQUINOL-30[c]", 
            "CPD-9956[c]", 
            "OCTAPRENYL-METHYL-OH-METHOXY-BENZQ[c]", 
            "CPD-9861[c]", 
            "CPD-9955[c]", 
            "UBIQUINONE-8[c]", 
            "UBIQUINONE-6[c]"]

for m in notubi10:
    print("==========\nMETABOLITE: " + m)
    _ = get_metabolite_reactions(m)

METABOLITE: UBIQUINOL-30[c]
Consuming reactions:
Producing reactions:
	RXN0-5258-Yeast: GLYCEROL-3P[c] + UBIQUINONE-6[c] --> DIHYDROXY-ACETONE-PHOSPHATE[c] + UBIQUINOL-30[c]
	RXN0-5330-YEAST: NADH[c] + PROTON[c] + UBIQUINONE-6[c] --> NAD[c] + UBIQUINOL-30[c]

METABOLITE: CPD-9956[c]
Consuming reactions:
Producing reactions:
	GLYC3PDEHYDROG-RXN: GLYCEROL-3P[c] + UBIQUINONE-8[c] --> CPD-9956[c] + DIHYDROXY-ACETONE-PHOSPHATE[c]

METABOLITE: OCTAPRENYL-METHYL-OH-METHOXY-BENZQ[c]
Consuming reactions:
Producing reactions:
	OCTAPRENYL-METHYL-METHOXY-BENZOQ-OH-RXN: NADH[c] + OCTAPRENYL-METHYL-METHOXY-BENZQ[c] + OXYGEN-MOLECULE[c] + PROTON[c] --> NAD[c] + OCTAPRENYL-METHYL-OH-METHOXY-BENZQ[c] + WATER[c]

METABOLITE: CPD-9861[c]
Consuming reactions:
	RXN-9229: CPD-9861[c] + S-ADENOSYLMETHIONINE[c] --> ADENOSYL-HOMO-CYS[c] + CPD-9955[c] + PROTON[c]
Producing reactions:

METABOLITE: CPD-9955[c]
Consuming reactions:
Producing reactions:
	RXN-9229: CPD-9861[c] + S-ADENOSYLMETHIONINE[c] --> ADENOSYL-

In [56]:
# Looks pretty barebones, let's see what happens if we remove them all
with model:
    model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)

    sol = model.optimize()
    print(f"Basal growth rate with: {sol.objective_value:.4f} 1/hr")
    
    for m in notubi10:
        reactions = [item for sublist in get_metabolite_reactions(m) for item in sublist]
        for r in reactions:
            r.knock_out()

    sol = model.optimize()
    print(f"Growth rate without non-ubiquinol 10 elements: {sol.objective_value:.4f} 1/hr")


Basal growth rate with: 0.5687 1/hr
Consuming reactions:
Producing reactions:
	RXN0-5258-Yeast: GLYCEROL-3P[c] + UBIQUINONE-6[c] --> DIHYDROXY-ACETONE-PHOSPHATE[c] + UBIQUINOL-30[c]
	RXN0-5330-YEAST: NADH[c] + PROTON[c] + UBIQUINONE-6[c] --> NAD[c] + UBIQUINOL-30[c]

Consuming reactions:
Producing reactions:
	GLYC3PDEHYDROG-RXN: GLYCEROL-3P[c] + UBIQUINONE-8[c] --> CPD-9956[c] + DIHYDROXY-ACETONE-PHOSPHATE[c]

Consuming reactions:
Producing reactions:
	OCTAPRENYL-METHYL-METHOXY-BENZOQ-OH-RXN: NADH[c] + OCTAPRENYL-METHYL-METHOXY-BENZQ[c] + OXYGEN-MOLECULE[c] + PROTON[c] --> NAD[c] + OCTAPRENYL-METHYL-OH-METHOXY-BENZQ[c] + WATER[c]

Consuming reactions:
	RXN-9229: CPD-9861[c] + S-ADENOSYLMETHIONINE[c] --> ADENOSYL-HOMO-CYS[c] + CPD-9955[c] + PROTON[c]
Producing reactions:

Consuming reactions:
Producing reactions:
	RXN-9229: CPD-9861[c] + S-ADENOSYLMETHIONINE[c] --> ADENOSYL-HOMO-CYS[c] + CPD-9955[c] + PROTON[c]

Consuming reactions:
	GLYC3PDEHYDROG-RXN: GLYCEROL-3P[c] + UBIQUINONE-8[c] 

Ok so we can remove the following reactions without issue:

["UBIQUINOL-30[c]", "CPD-9956[c]", "OCTAPRENYL-METHYL-OH-METHOXY-BENZQ[c]", "CPD-9861[c]", "CPD-9955[c]"]


In [55]:
# check ubiquinol 10 is indeed essential in our model
get_metabolite_reactions("CPD-9958[c]")
print("====")
_ = check_essential_metabolite("CPD-9958[c]")

Consuming reactions:
	RXN-20992: 2.0 CPD-9958[c] + OXYGEN-MOLECULE[c] --> 2.0 UBIQUINONE-10[c] + 2.0 WATER[c]
	1.10.2.2-RXN: CPD-9958[c] + 2.0 Cytochromes-C-Oxidized[p] --> 2.0 Cytochromes-C-Reduced[p] + 2.0 PROTON[c] + UBIQUINONE-10[c]
Producing reactions:
	RXN-14903: PRO[c] + UBIQUINONE-10[c] --> CPD-9958[c] + L-DELTA1-PYRROLINE_5-CARBOXYLATE[c] + PROTON[c]
	SUCCINATE-DEHYDROGENASE-UBIQUINONE-RXN: SUC[c] + UBIQUINONE-10[c] <=> CPD-9958[c] + FUM[c]
	1.5.5.1-RXN-ETF-Reduced/UBIQUINONE-10//ETF-Oxidized/CPD-9958/PROTON.56.: ETF-Reduced[c] + UBIQUINONE-10[c] --> CPD-9958[c] + ETF-Oxidized[c] + PROTON[c]
	RXN-9237: CPD-9873[c] + S-ADENOSYLMETHIONINE[c] --> ADENOSYL-HOMO-CYS[c] + CPD-9958[c] + PROTON[c]
	NADH-DEHYDROG-A-RXN: NADH[c] + 5.0 PROTON[c] + UBIQUINONE-10[c] <=> CPD-9958[c] + NAD[c] + 4.0 PROTON[e]

====
Growth rate with CPD-9958[c]: 0.5687 1/hr
Growth rate without CPD-9958[c]: 0.4785 1/hr


Ubiquinol 10 is not essential. This does not sound right, but we will investigate this more

Let's next investigate the genes from the essential gene review again which were linked to ubiquinol 8 synthesis.

## 1. RXN-8992

In [34]:
rxn_id = "RXN-8992"
rxn = model.reactions.get_by_id(rxn_id)
print(f"{rxn.id}:\n\t{rxn.reaction}\n")

RXN-8992:
	5.0 DELTA3-ISOPENTENYL-PP[c] + FARNESYL-PP[c] --> OCTAPRENYL-DIPHOSPHATE[c] + 5.0 PPI[c]



In [40]:
m = model.metabolites.get_by_id("OCTAPRENYL-DIPHOSPHATE[c]")
print(m.id, m.name, m.formula)

OCTAPRENYL-DIPHOSPHATE[c] all-trans-octaprenyl diphosphate C40H65O7P2


In [38]:
get_metabolite_reactions("OCTAPRENYL-DIPHOSPHATE[c]")
get_metabolite_reactions("3-OCTAPRENYL-4-HYDROXYBENZOATE[c]")
get_metabolite_reactions("2-OCTAPRENYLPHENOL[c]")
get_metabolite_reactions("2-OCTAPRENYL-6-HYDROXYPHENOL[c]")

Consuming reactions:
	4OHBENZOATE-OCTAPRENYLTRANSFER-RXN: 4-hydroxybenzoate[c] + OCTAPRENYL-DIPHOSPHATE[c] --> 3-OCTAPRENYL-4-HYDROXYBENZOATE[c] + PPI[c]
Producing reactions:
	RXN-8992: 5.0 DELTA3-ISOPENTENYL-PP[c] + FARNESYL-PP[c] --> OCTAPRENYL-DIPHOSPHATE[c] + 5.0 PPI[c]

Consuming reactions:
	3-OCTAPRENYL-4-OHBENZOATE-DECARBOX-RXN: 3-OCTAPRENYL-4-HYDROXYBENZOATE[c] + PROTON[c] --> 2-OCTAPRENYLPHENOL[c] + CARBON-DIOXIDE[c]
Producing reactions:
	4OHBENZOATE-OCTAPRENYLTRANSFER-RXN: 4-hydroxybenzoate[c] + OCTAPRENYL-DIPHOSPHATE[c] --> 3-OCTAPRENYL-4-HYDROXYBENZOATE[c] + PPI[c]

Consuming reactions:
	2-OCTAPRENYLPHENOL-HYDROX-RXN: 2-OCTAPRENYLPHENOL[c] + NADPH[c] + OXYGEN-MOLECULE[c] + PROTON[c] --> 2-OCTAPRENYL-6-HYDROXYPHENOL[c] + NADP[c] + WATER[c]
Producing reactions:
	3-OCTAPRENYL-4-OHBENZOATE-DECARBOX-RXN: 3-OCTAPRENYL-4-HYDROXYBENZOATE[c] + PROTON[c] --> 2-OCTAPRENYLPHENOL[c] + CARBON-DIOXIDE[c]

Consuming reactions:
	COFACTOR-RXN: 0.14627036 2-OCTAPRENYL-6-HYDROXYPHENOL[c] + 0.3

([<Reaction COFACTOR-RXN at 0x130fef400>,
  <Reaction 2-OCTAPRENYL-6-OHPHENOL-METHY-RXN at 0x1163052b0>],
 [<Reaction 2-OCTAPRENYLPHENOL-HYDROX-RXN at 0x13197fee0>])

In [72]:
m = model.metabolites.get_by_id("2-OCTAPRENYL-6-HYDROXYPHENOL[c]")
print(m.id, m.name, m.formula)

2-OCTAPRENYL-6-HYDROXYPHENOL[c] 3-(all-trans-octaprenyl)benzene-1,2-diol C46H70O2


4-hydroxybenzoate[c] + OCTAPRENYL-DIPHOSPHATE[c] -> 3-OCTAPRENYL-4-HYDROXYBENZOATE[c] 

3-OCTAPRENYL-4-HYDROXYBENZOATE[c] + PROTON[c] --> 2-OCTAPRENYLPHENOL[c] + CARBON-DIOXIDE[c]

2-OCTAPRENYLPHENOL[c] + NADPH[c] + OXYGEN-MOLECULE[c] + PROTON[c] --> **2-OCTAPRENYL-6-HYDROXYPHENOL[c]** + NADP[c] + WATER[c]

2-OCTAPRENYL-6-HYDROXYPHENOL[c] feeds into required COFACTOR reaction


In [ ]:
get_metabolite_reactions("2-OCTAPRENYL-6-HYDROXYPHENOL[c]")
print("=====")
get_metabolite_reactions("2-OCTAPRENYL-6-METHOXYPHENOL[c]")
print("=====")
get_metabolite_reactions("OCTAPRENYL-METHOXY-BENZOQUINONE[c]")
print("=====")
get_metabolite_reactions("OCTAPRENYL-METHYL-METHOXY-BENZQ[c]")
print("=====")
get_metabolite_reactions("OCTAPRENYL-METHYL-OH-METHOXY-BENZQ[c]")

Consuming reactions:
	COFACTOR-RXN: 0.14627036 2-OCTAPRENYL-6-HYDROXYPHENOL[c] + 0.377810437 CO-A[c] + 0.14627036 FAD[c] + 0.14627036 FORMYL-THF-GLU-N[c] + 0.14627036 METHYLENE-THF-GLU-N[c] + 0.14627036 PROTOHEME[c] + 0.14627036 PYRIDOXAL_PHOSPHATE[c] + 0.14627036 RIBOFLAVIN[c] + 0.14627036 S-ADENOSYLMETHIONINE[c] + 0.14627036 THF-GLU-N[c] + 0.14627036 THIAMINE-PYROPHOSPHATE[c] --> COFACTORS[c]
	2-OCTAPRENYL-6-OHPHENOL-METHY-RXN: 2-OCTAPRENYL-6-HYDROXYPHENOL[c] + S-ADENOSYLMETHIONINE[c] --> 2-OCTAPRENYL-6-METHOXYPHENOL[c] + ADENOSYL-HOMO-CYS[c] + PROTON[c]
Producing reactions:
	2-OCTAPRENYLPHENOL-HYDROX-RXN: 2-OCTAPRENYLPHENOL[c] + NADPH[c] + OXYGEN-MOLECULE[c] + PROTON[c] --> 2-OCTAPRENYL-6-HYDROXYPHENOL[c] + NADP[c] + WATER[c]

=====
Consuming reactions:
	2-OCTAPRENYL-6-METHOXYPHENOL-HYDROX-RXN: 2-OCTAPRENYL-6-METHOXYPHENOL[c] + NADPH[c] + OXYGEN-MOLECULE[c] + PROTON[c] --> NADP[c] + OCTAPRENYL-METHOXY-BENZOQUINONE[c] + WATER[c]
Producing reactions:
	2-OCTAPRENYL-6-OHPHENOL-METHY-RXN

([], [<Reaction OCTAPRENYL-METHYL-METHOXY-BENZOQ-OH-RXN at 0x131f149d0>])

In [73]:
m = model.metabolites.get_by_id("OCTAPRENYL-METHYL-OH-METHOXY-BENZQ[c]")
print(m.id, m.name, m.formula)

OCTAPRENYL-METHYL-OH-METHOXY-BENZQ[c] 3-demethylubiquinol-8 C48H74O4


In [49]:
get_metabolite_reactions("UBIQUINONE-8[c]")

Consuming reactions:
	GLYC3PDEHYDROG-RXN: GLYCEROL-3P[c] + UBIQUINONE-8[c] --> CPD-9956[c] + DIHYDROXY-ACETONE-PHOSPHATE[c]
Producing reactions:



([<Reaction GLYC3PDEHYDROG-RXN at 0x13193feb0>], [])

The ubiquinol 10 pathway uses decaprenyl instead of octaprenyl. (CPD-9610[c])

In [57]:
search_metabolites(["decaprenyl"])

id                          | name                                                                                                                                                                          | formula       
----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------
UNDECAPRENYL-DIPHOSPHATE[c] | di-trans,octa-cis-undecaprenyl diphosphate                                                                                                                                    | C55H89O7P2    
C5[c]                       | undecaprenyldiphospho-N-acetylmuramoyl-L-alanyl-gamma-D-glutamyl-meso-2,6-diaminopimeloyl-D-alanyl-D-alanine                                                                  | C87H139N7O23P2
C6[c]                       | ditrans,octacis-undecaprenyldiphospho-[(N-acetyl-beta-D-glucosaminyl)-(1,4)-]-N-acetyl

In [63]:
get_metabolite_reactions("CPD-9610[c]")

print("=====")

m = model.metabolites.get_by_id("CPD-9864[c]")
print(m.id, m.name, m.formula)
get_metabolite_reactions("CPD-9864[c]")

print("=====")

# we don't have the rest of the ubiquinol 10 producing pathway 

Consuming reactions:
	RXN-9230: 4-hydroxybenzoate[c] + CPD-9610[c] --> CPD-9864[c] + PPI[c]
Producing reactions:
	RXN-9106: 7.0 DELTA3-ISOPENTENYL-PP[c] + FARNESYL-PP[c] --> CPD-9610[c] + 7.0 PPI[c]

=====
CPD-9864[c] 3-decaprenyl-4-hydroxybenzoate C57H85O3
Consuming reactions:
Producing reactions:
	RXN-9230: 4-hydroxybenzoate[c] + CPD-9610[c] --> CPD-9864[c] + PPI[c]

=====


In [66]:
get_metabolite_reactions("CPD-9862[c]")

Consuming reactions:
	RXN-9232: CPD-9862[c] + NADPH[c] + OXYGEN-MOLECULE[c] + PROTON[c] --> CPD-9867[c] + NADP[c] + WATER[c]
Producing reactions:



([<Reaction RXN-9232 at 0x130f76be0>], [])

In [67]:
get_metabolite_reactions("CPD-9867[c]")

Consuming reactions:
Producing reactions:
	RXN-9232: CPD-9862[c] + NADPH[c] + OXYGEN-MOLECULE[c] + PROTON[c] --> CPD-9867[c] + NADP[c] + WATER[c]



([], [<Reaction RXN-9232 at 0x130f76be0>])

NEED TO ADD THE BELOW REACTION

CPD-9864[c] + PROTON[c] -> CPD-9862[c] + CARBON-DIOXIDE[c]



In [77]:
search_metabolites(["CPD-9871[c]"])

id | name | formula
---+------+--------


In [71]:
search_metabolites(["decaprenyl"])

id                          | name                                                                                                                                                                          | formula       
----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------
UNDECAPRENYL-DIPHOSPHATE[c] | di-trans,octa-cis-undecaprenyl diphosphate                                                                                                                                    | C55H89O7P2    
C5[c]                       | undecaprenyldiphospho-N-acetylmuramoyl-L-alanyl-gamma-D-glutamyl-meso-2,6-diaminopimeloyl-D-alanyl-D-alanine                                                                  | C87H139N7O23P2
C6[c]                       | ditrans,octacis-undecaprenyldiphospho-[(N-acetyl-beta-D-glucosaminyl)-(1,4)-]-N-acetyl

# Implementing the Ubiquinol-10 Pathway

In [114]:
get_metabolite_reactions("CPD-9867[c]")

Consuming reactions:
Producing reactions:
	RXN-9232: CPD-9862[c] + NADPH[c] + OXYGEN-MOLECULE[c] + PROTON[c] --> CPD-9867[c] + NADP[c] + WATER[c]



([], [<Reaction RXN-9232 at 0x130f76be0>])

In [130]:
with model:
    # main metabolites we need
    CPD_9864_c = model.metabolites.get_by_id("CPD-9864[c]")
    CPD_9862_c = model.metabolites.get_by_id("CPD-9862[c]")
    CPD_9867_c = model.metabolites.get_by_id("CPD-9867[c]")
    
    CPD_9865_c = cobra.Metabolite(
        'CPD-9865[c]',
        formula='C57H88O2',
        name="2-methoxy-6-(all-trans-decaprenyl)phenol",
        compartment='c')

    CPD_9869_c = cobra.Metabolite(
        'CPD-9869[c]',
        formula='C57H88O3',
        name="ArithmeticError2-methoxy-6-(all-trans-decaprenyl)benzene-1,4-diol",
        compartment='c')
        
    CPD_9871_c = cobra.Metabolite(
        'CPD-9871[c]',
        formula='C58H90O3',
        name="5-methoxy-2-methyl-3-(all-trans-decaprenyl)benzene-1,4-diol",
        compartment='c')
    
    CPD_9873_c = model.metabolites.get_by_id("CPD-9873[c]")
    
    model.add_metabolites([CPD_9865_c, CPD_9869_c, CPD_9871_c])
    
    # other metabolites 
    PROTON = model.metabolites.get_by_id("PROTON[c]")
    WATER = model.metabolites.get_by_id("WATER[c]")
    CARBON_DIOXIDE = model.metabolites.get_by_id("CARBON-DIOXIDE[c]")
    S_ADENOSYLMETHIONINE = model.metabolites.get_by_id("S-ADENOSYLMETHIONINE[c]")
    ADENOSYL_HOMO_CYS = model.metabolites.get_by_id("ADENOSYL-HOMO-CYS[c]")
    OXYGEN_MOLECULE = model.metabolites.get_by_id("OXYGEN-MOLECULE[c]")
    NADPH = model.metabolites.get_by_id("NADPH[c]")
    NADP = model.metabolites.get_by_id("NADP[c]")
    NADH = model.metabolites.get_by_id("NADH[c]")
    NAD = model.metabolites.get_by_id("NAD[c]")
    
    # reaction addition
    rxn4_1_1_98 = cobra.Reaction("RXN_4.1.1.98")
    rxn4_1_1_98.add_metabolites({CPD_9864_c: -1, PROTON: -1, CPD_9862_c: 1, CARBON_DIOXIDE: 1})
    
    rxn2_1_1_222 = cobra.Reaction("RXN_2.1.1.222")
    rxn2_1_1_222.add_metabolites({CPD_9867_c: -1, S_ADENOSYLMETHIONINE: -1, CPD_9865_c: 1, ADENOSYL_HOMO_CYS: 1, PROTON: 1})
    
    rxn1_14_13_M56 = cobra.Reaction("RXN_1.14.13.M56")
    rxn1_14_13_M56.add_metabolites({CPD_9865_c: -1, PROTON: -1, NADPH: -1, OXYGEN_MOLECULE: -1, CPD_9869_c: 1, NADP: 1, WATER: 1})
    
    rxn2_1_1_201 = cobra.Reaction("RXN_2.1.1.201")
    rxn2_1_1_201.add_metabolites({CPD_9869_c: -1, S_ADENOSYLMETHIONINE: -1, CPD_9871_c: 1, ADENOSYL_HOMO_CYS: 1, PROTON: 1})
    
    rxn1_14_99_60_NADH = cobra.Reaction("RXN_1.14.99.60_NADH")
    rxn1_14_99_60_NADH.add_metabolites({CPD_9871_c:-1, NADH:-1, OXYGEN_MOLECULE:-1, CPD_9873_c: 1, NAD: 1, WATER: 1})
    
    rxn1_14_99_60_NADPH = cobra.Reaction("RXN_1.14.99.60_NADPH")
    rxn1_14_99_60_NADPH.add_metabolites({CPD_9871_c:-1, NADPH:-1, OXYGEN_MOLECULE:-1, CPD_9873_c: 1, NADP: 1, WATER: 1})
    
    model.add_reactions([rxn4_1_1_98, rxn2_1_1_222, rxn1_14_13_M56, rxn2_1_1_201, rxn1_14_99_60_NADH, rxn1_14_99_60_NADPH])

    # fix the COFACTOR reaction 
    model.reactions.get_by_id("COFACTOR-RXN").knock_out()
 
    # metabolites
    CO_A = model.metabolites.get_by_id("CO-A[c]")
    FAD = model.metabolites.get_by_id("FAD[c]")
    FORMYL_THF = model.metabolites.get_by_id("FORMYL-THF-GLU-N[c]")
    METHYLENE_THF = model.metabolites.get_by_id("METHYLENE-THF-GLU-N[c]")
    PROTOHEME = model.metabolites.get_by_id("PROTOHEME[c]")
    PYRIDOXAL = model.metabolites.get_by_id("PYRIDOXAL_PHOSPHATE[c]")
    RIBOFLAVIN = model.metabolites.get_by_id("RIBOFLAVIN[c]")
    SAM = model.metabolites.get_by_id("S-ADENOSYLMETHIONINE[c]")
    THF_GLU = model.metabolites.get_by_id("THF-GLU-N[c]")
    TPP = model.metabolites.get_by_id("THIAMINE-PYROPHOSPHATE[c]")
    COFACTORS = model.metabolites.get_by_id("COFACTORS[c]")

    COFACTOR_RXN = cobra.Reaction("COFACTOR-RXN2")
    COFACTOR_RXN.add_metabolites({
        CPD_9867_c: -0.14627036,
        CO_A: -0.377810437,
        FAD: -0.14627036,
        FORMYL_THF: -0.14627036,
        METHYLENE_THF: -0.14627036,
        PROTOHEME: -0.14627036,
        PYRIDOXAL: -0.14627036,
        RIBOFLAVIN: -0.14627036,
        SAM: -0.14627036,
        THF_GLU: -0.14627036,
        TPP: -0.14627036,
        COFACTORS: 1
    })
    model.add_reactions([COFACTOR_RXN])
    # delete old ubquinol 6 reactions
    ubi6syn_reactions = ["4OHBENZOATE-OCTAPRENYLTRANSFER-RXN", 
                         "3-OCTAPRENYL-4-OHBENZOATE-DECARBOX-RXN", 
                         "2-OCTAPRENYLPHENOL-HYDROX-RXN",
                         "2-OCTAPRENYL-6-OHPHENOL-METHY-RXN",
                         "2-OCTAPRENYL-6-METHOXYPHENOL-HYDROX-RXN",
                         "2-OCTAPRENYL-METHOXY-BENZOQ-METH-RXN",
                         "OCTAPRENYL-METHYL-METHOXY-BENZOQ-OH-RXN"]
    print("===Removing  Ubiquinol 6 Reactions===")
    for rxn in ubi6syn_reactions:
        print("removing: ", rxn)
        model.reactions.get_by_id(rxn).knock_out()
    
    print("\n===Removing Additional Non-Ubiquinol 10 Reactions===")
    for m in notubi10:
        reactions = [item for sublist in get_metabolite_reactions(m, model=model, verbose=False) for item in sublist]
        for r in reactions:
            print("removing: ", r.id)
            r.knock_out()
    
    model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)
    sol = model.optimize()
    print(f"\nGrowth rate with {sol.objective_value:.4f} 1/hr")

===Removing  Ubiquinol 6 Reactions===
removing:  4OHBENZOATE-OCTAPRENYLTRANSFER-RXN
removing:  3-OCTAPRENYL-4-OHBENZOATE-DECARBOX-RXN
removing:  2-OCTAPRENYLPHENOL-HYDROX-RXN
removing:  2-OCTAPRENYL-6-OHPHENOL-METHY-RXN
removing:  2-OCTAPRENYL-6-METHOXYPHENOL-HYDROX-RXN
removing:  2-OCTAPRENYL-METHOXY-BENZOQ-METH-RXN
removing:  OCTAPRENYL-METHYL-METHOXY-BENZOQ-OH-RXN

===Removing Additional Non-Ubiquinol 10 Reactions===
removing:  RXN0-5330-YEAST
removing:  RXN0-5258-Yeast
removing:  GLYC3PDEHYDROG-RXN
removing:  OCTAPRENYL-METHYL-METHOXY-BENZOQ-OH-RXN
removing:  RXN-9229
removing:  RXN-9229
removing:  GLYC3PDEHYDROG-RXN
removing:  RXN0-5330-YEAST
removing:  RXN0-5258-Yeast

Growth rate with 0.5687 1/hr


## Figuring out 4-Hydroxbenzoate synthesis

I think we have the tyrosine -> coumarate -> 4 hydroxybenzoate pathway.  

See https://www.frontiersin.org/journals/bioengineering-and-biotechnology/articles/10.3389/fbioe.2019.00130/full

We have an 88% hit in BLAST between Tyrosine ammonia-lyase in Streptomyces globisporus with R Pom's Histidine ammonia-lyase enzyme. This high sequence similarity, might indicate to me that tryosine can also react with this enzyme.

Once we have coumarate, we can make p-coumaryl and use the coumaryl pathway for 4-hydroxybenzoate synthesis. 

We can add reactions 3.1.2.20 and 3.1.6.1 from biocyc to complete this pathway:

1. dimethylsulfoniopropanoate + 4-hydroxybenzoyl-CoA → dimethylsulfoniopropioyl-CoA + 4-hydroxybenzoate

2. 4-hydroxybenzoyl-CoA + H2O → coenzyme A + 4-hydroxybenzoate + H+        




In [164]:
with model:
    model.reactions.get_by_id("CHORPYRLY-RXN").knock_out()
    
    # define relevant metabolites
    DMSP = model.metabolites.get_by_id("SS-DIMETHYL-BETA-PROPIOTHETIN[c]")
    HBCOA = model.metabolites.get_by_id("CPD-201[c]")
    DMSPCOA = model.metabolites.get_by_id("CPD-10473[c]")
    HB = model.metabolites.get_by_id("4-hydroxybenzoate[c]")
    COA = model.metabolites.get_by_id("CO-A[c]")
    PROTON = model.metabolites.get_by_id("PROTON[c]")
    WATER = model.metabolites.get_by_id("WATER[c]")
    COUMARATE = model.metabolites.get_by_id("COUMARATE[c]")
    TYROSINE = model.metabolites.get_by_id("TYR[c]")
    AMMONIUM = model.metabolites.get_by_id("AMMONIUM[c]")
    dimethylsulfoniopropanoate = model.metabolites.get_by_id("SS-DIMETHYL-BETA-PROPIOTHETIN[c]")
    dimethylsulfoniopropioyl_CoA = model.metabolites.get_by_id("CPD-10473[c]") 
    hydroxybenzoyl_CoA = model.metabolites.get_by_id("CPD-201[c]")
    
    # define reaction 4.3.1.23
    rxn_4_3_1_23 = cobra.Reaction("RXN_4.3.1.23")
    rxn_4_3_1_23.add_metabolites({TYROSINE: -1, COUMARATE: 1, AMMONIUM: 1})
            
    # define rection 3.1.2.20
    #1. dimethylsulfoniopropanoate + 4-hydroxybenzoyl-CoA → dimethylsulfoniopropioyl-CoA + 4-hydroxybenzoate
    rxn_3_1_2_20 = cobra.Reaction("RXN_3.1.2.20")
    rxn_3_1_2_20.add_metabolites({dimethylsulfoniopropanoate: -1, hydroxybenzoyl_CoA: -1, HB: 1, dimethylsulfoniopropioyl_CoA: 1})
    
    # define reaction 3.1.6.1
    #2. 4-hydroxybenzoyl-CoA + H2O → coenzyme A + 4-hydroxybenzoate + H+    
    rxn_3_1_6_1 = cobra.Reaction("RXN_3.1.6.1")
    rxn_3_1_6_1.add_metabolites({hydroxybenzoyl_CoA: -1, WATER: -1, COA: 1, HB: 1, PROTON: 1})    
    
    model.add_reactions([rxn_4_3_1_23, rxn_3_1_2_20, rxn_3_1_6_1])
    
    # test model
    model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)
    sol = model.optimize()
    print(f"Growth rate with {sol.objective_value:.4f} 1/hr")

Growth rate with 0.5687 1/hr
